# Génération d’un dataset industriel synthétique contexte GDIZ Bénin

## Objectif

Ce notebook construire un dataset de maintenance prédictive inspiré de [AI4I 2020](https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset) et l’adapter à un contexte industriel réaliste (type [GDIZ Bénin](https://gdiz-benin.com/fr/)).

Cela passe par l’introduction de :
- contraintes énergétiques
- conditions environnementales
- usure machine réaliste
- pannes probabilistes

Donc, la base de données sera un ensemble de 10 000 enregistrements simulant le comportement de machines industrielles à la GDIZ, où chaque ligne combine des paramètres physiques (vitesse, couple) et des facteurs environnementaux critiques du Bénin (humidité, poussière, instabilité électrique).

In [ ]:
import numpy as np
import pandas as pd
import json

## Génération des caractéristiques machines

On introduit 3 types de machines (L, M, H) où le type influence directement le stress mécanique.

In [2]:
n = 10000
np.random.seed(1)

machine_type = np.random.choice(['L', 'M', 'H'], n, p=[0.5, 0.3, 0.2]) # L : low, M : medium, H : high quality machines

load_factor_map = {'L': 0.8, 'M': 1.0, 'H': 1.2}
load_factor = np.array([load_factor_map[t] for t in machine_type])

## Features

### Environnement

La chaleur et l’humidité sont liées de manière inverse (*plus il fait chaud, moins il y a d’humidité*). 
La poussière suit une distribution asymétrique avec des pics réalistes (*par ex, plus de poussière pendant la saison sèche*).

In [3]:
# Saisonnalité (0 = saison des pluies, 1 = saison sèche, avec une probabilité de 70% pour la saison sèche)
season = np.random.binomial(1, 0.7, n)

ambient_temp = np.where(
    season == 1,
    34 + np.random.normal(0, 1.5, n), # Chaud et stable
    30 + np.random.normal(0, 2.5, n)  # Plus frais et variable
)
ambient_temp = np.clip(ambient_temp, 23, 44)

# Pluie (Plus probable en saison == 0)
prob_rain = np.where(season == 0, 0.4, 0.1) 
rain_flag = (np.random.rand(n) < prob_rain).astype(int)

humidity = (88 - (ambient_temp - 30) * 2.5 + rain_flag * 8 + np.random.normal(0, 4, n))
humidity = np.clip(humidity, 50, 98)

dust = np.random.gamma(2, 10, n)

### Energie

In [4]:
power_outage = np.random.binomial(1, 0.05, n)
# Ce n’est pas le voltage moyen qui compte, mais les événements extrêmes
voltage = np.where(
    power_outage == 1,
    np.random.normal(180, 20, n),
    np.random.normal(220, 5, n)
)

voltage_stability = 1 - np.abs(voltage - 220) / 50
voltage_stability = np.clip(voltage_stability, 0, 1)

### Capteurs machines physiques

Si la charge augmente, la température augmente également. De plus, le type de machine influence le couple (`torque`) généré.

In [5]:
speed = np.random.normal(1500, 150, n)
# Une machine H (load 1.2) est plus efficace, une machine L (load 0.8) peine plus et monte en couple.
torque = np.random.normal(40, 10, n) * (1 / load_factor) 
torque = np.clip(torque, 0, 80)

machine_load = np.clip(load_factor + np.random.normal(0, 0.1, n), 0.5, 1.5)

# La machine chauffe plus à haute charge et en saison sèche
process_temp = ambient_temp + 10 + 5*machine_load + 2*(machine_load**2) + np.random.normal(0, 1, n)

### Usure

Lorsque les machines fonctionnent, elles subissent une usure naturelle qui s’accumule au fil du temps. Donc, il est courant que des pièces soient remplacées de manière aléatoire pour maintenir les machines en bon état de fonctionnement. Cependant, la maintenance peut être imparfaite, ce qui signifie que même après une intervention, la machine peut ne pas être restaurée à son état optimal.

In [6]:
tool_wear = np.zeros(n)
current_wear = np.random.uniform(0, 50) 
# On définit le seuil de survie du premier outil
current_threshold = np.random.uniform(180, 250) 

for i in range(n):
    # Calcul de l'usure a ajouter (Charge + Poussière + Aléatoire)
    increment = (
        0.02 * machine_load[i] + 
        0.01 * (dust[i] / 50) + 
        np.random.normal(0.01, 0.005)
    )
    current_wear += increment

    if current_wear > current_threshold:
        current_wear = np.random.uniform(0, 5) # Nouvelle Machine
        current_threshold = np.random.uniform(180, 250) # Nouveau seuil pour remplcement

    tool_wear[i] = current_wear

### Chocs systèmes et redémarrages

Les pannes arrivent souvent au redémarrage (la machine est plus fragile au démarrage).

In [7]:
restart_shock = power_outage * np.random.binomial(1, 0.3, n)

### Vibrations

In [8]:
vibration = (
    0.2 +
    0.003 * tool_wear +
    0.2 * machine_load +
    0.5 * (1 - voltage_stability) +
    restart_shock * np.random.uniform(0.5, 1.5, n) +
    np.random.normal(0, 0.05, n)
)

## Construction du stress mécanique

Une panne n'est pas un évenement isolé, mais le résultat d'une accumulation de stress mécanique. Par exemple, une machine qui fonctionne à haute charge dans un environnement poussiéreux et humide subira plus de stress que si elle fonctionnait à faible charge dans un environnement propre et sec.

In [9]:
# Normalisation des facteurs (0 à 1)
n_wear = tool_wear / 250
n_load = machine_load / 1.5
n_temp = (process_temp - ambient_temp) / 20 # Gradient thermique

stress = (
    0.6 * (n_wear * n_load) +             
    0.3 * (n_temp) +                      
    0.5 * (1 - voltage_stability) +      
    0.2 * (dust * humidity / 5000)        
)

In [10]:
target_rate = 0.04
# Contrôle du nombre de pannes
stress_norm = (stress - np.mean(stress)) / np.std(stress)
threshold = np.percentile(stress_norm, 100 * (1 - target_rate))
target = (stress_norm > threshold).astype(int)

In [11]:
# 4% de pannes logiques + 0.5% de pannes purement aléatoires
random_failure = np.random.binomial(1, 0.005, n)
target = np.clip(target + random_failure, 0, 1)

## Dataset

In [12]:
df = pd.DataFrame({
    "season": season,
    "machine_type": machine_type,
    "ambient_temperature": ambient_temp,
    "process_temperature": process_temp,
    "rotational_speed": speed,
    "torque": torque,
    "tool_wear": tool_wear,
    "machine_load": machine_load,
    "vibration": vibration,
    "humidity": humidity,
    "dust": dust,
    "rain_flag": rain_flag,
    "power_outage": power_outage,
    "voltage": voltage,
    "voltage_stability": voltage_stability,
    "target": target
})

In [15]:
print(df.head())

   season machine_type  ambient_temperature  process_temperature  \
0       0            L            32.034346            47.652070   
1       1            M            36.064876            55.387924   
2       0            L            29.595425            44.670585   
3       0            L            32.242163            47.257286   
4       1            L            33.326884            50.016889   

   rotational_speed     torque  tool_wear  machine_load  vibration   humidity  \
0       1654.363226  44.930872  21.125454      0.799123   0.418343  89.702547   
1       1475.001280  40.090857  21.168451      1.135719   0.549210  67.802706   
2       1556.266021  64.101911  21.210642      0.838211   0.456233  98.000000   
3       1576.524801  56.471916  21.249738      0.864494   0.399467  90.836644   
4       1730.390249  48.258781  21.270749      0.853141   0.468895  81.934127   

        dust  rain_flag  power_outage     voltage  voltage_stability  target  
0  21.754830          1  

## Dictionnaire des données

In [26]:
metadata = {
  "dataset_info": {
    "name": "GDIZ Predictive Maintenance Dataset",
    "version": "1.0",
    "total_rows": 10000,
    "target_rate": 0.04
  },
  "variables": [
    {
      "name": "season",
      "type": "integer",
      "unit": "category",
      "description": "Saison actuelle (0 : Pluies, 1 : Saison sèche/Harmattan)."
    },
    {
      "name": "machine_type",
      "type": "string",
      "unit": "category",
      "description": "Qualité de la machine (L: Entrée de gamme, M: Moyenne, H: Haute qualité)."
    },
    {
      "name": "ambient_temperature",
      "type": "float",
      "unit": "°C",
      "description": "Température de l'air ambiant dans l'usine."
    },
    {
      "name": "process_temperature",
      "type": "float",
      "unit": "°C",
      "description": "Température interne de fonctionnement de la machine."
    },
    {
      "name": "rotational_speed",
      "type": "float",
      "unit": "rpm",
      "description": "Vitesse de rotation de l'outil moteur."
    },
    {
      "name": "torque",
      "type": "float",
      "unit": "Nm",
      "description": "Couple de force exercé par la machine."
    },
    {
      "name": "tool_wear",
      "type": "float",
      "unit": "min",
      "description": "Temps d'usure cumulé de l'outil actuel."
    },
    {
      "name": "machine_load",
      "type": "float",
      "unit": "ratio",
      "description": "Taux de charge de travail appliqué à la machine."
    },
    {
      "name": "vibration",
      "type": "float",
      "unit": "mm/s",
      "description": "Intensité des vibrations mécaniques détectées."
    },
    {
      "name": "humidity",
      "type": "float",
      "unit": "%",
      "description": "Taux d'humidité relative dans l'environnement."
    },
    {
      "name": "dust",
      "type": "float",
      "unit": "µg/m³",
      "description": "Niveau de poussière et de particules dans l'air."
    },
    {
      "name": "rain_flag",
      "type": "integer",
      "unit": "boolean",
      "description": "Présence de pluie (1) ou non (0)."
    },
    {
      "name": "power_outage",
      "type": "integer",
      "unit": "boolean",
      "description": "Indicateur de coupure de courant soudaine."
    },
    {
      "name": "voltage",
      "type": "float",
      "unit": "V",
      "description": "Tension électrique mesurée à l'entrée de la machine."
    },
    {
      "name": "voltage_stability",
      "type": "float",
      "unit": "ratio",
      "description": "Indice de stabilité du réseau électrique (1.0 = stable)."
    },
    {
      "name": "target",
      "type": "integer",
      "unit": "boolean",
      "description": "État final de la machine : 0 (Normal) ou 1 (Panne)."
    }
  ]
}

## Sauvegarde

In [ ]:
# Sauvegarde des données en .csv et .parquet
df.to_parquet('data/input/gdiz_dataset.parquet', index=False)
df.to_csv('data/input/gdiz_dataset.csv', index=False)

In [ ]:
# Ecriture du dictionnaire de variables
with open('data/input/metadata.json', 'w') as f:
    json.dump(metadata, f)